In [49]:
# libraries imports

from agents import Agent, trace, Runner, WebSearchTool, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, EmailStr, Field
from dotenv import load_dotenv
import smtplib
from email.message import EmailMessage
import os
import asyncio
from IPython.display import display, Markdown



In [38]:
#load environmental variables
load_dotenv(override = True)
gmail_username = os.getenv("GMAIL_USERNAME")
gmail_password = os.getenv("GMAIL_PASSWORD")
openai_api_key = os.getenv("OPENAI_API_KEY")

In [39]:
# send email function/tool

@function_tool
def send_email(subject: str, html_body: str):
    """ This function is used to send emails to the end user. """
    subject = subject
    html_body = html_body

    mail = EmailMessage()
    mail["Subject"] = subject
    mail["To"] = "mazharotagou@gmail.com"
    mail["From"] = "botmazhar8@gmail.com"
    mail.add_alternative(html_body, subtype = "html")

    with smtplib.SMTP_SSL('smtp.gmail.com', 465) as smtp:
        smtp.login(gmail_username, gmail_password)
        response = smtp.send_message(mail)
        if response == {}:
            return {"role":"tool","email_status":"successful"}
        else:
            return {"role":"tool","email_status":"failed"}


In [41]:
import asyncio
#Check email
checkmail_instructions = f""" You are an agent whose task is to send the contents recieved using send_email tool"""
check_agent = Agent(name = "check_agent", instructions = checkmail_instructions, tools = [send_email], model = "gpt-4o-mini")

message = f""" Subject of the email: What are you doing?
                Body of the email: I am doing alright! """


async def main():
    with trace("check email"):
        result = await Runner.run(check_agent, message)
        print (result.final_output)

await main()

The email has been successfully sent with the subject "What are you doing?" and the message "I am doing alright!"


In [52]:
HOW_MANY_SEARCHES = 3

INSTRUCTIONS = f"You are a helpful research assistant. Given a query, come up with a set of web searches \
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for."



#Websearch Planner
class WebSearchItem(BaseModel):
    query: str = Field(description = "The search term to be used for websearch.")
    reason: str = Field(description ="The reasoning for why this search is important to the query.")

class WebSearchPlanner(BaseModel):
    searches : list[WebSearchItem] = Field(description = "The list of websearches to perform to best answer the query.")

planner_agent = Agent(
    name ="PlannerAgent",
    instructions = INSTRUCTIONS,
    model = "gpt-4o-mini",
    output_type= WebSearchPlanner,
    )
    

In [45]:
message = "ANU's WHS Framework"

with trace("Search"):
    result = await Runner.run(planner_agent, message)
    print(result.final_output)

searches=[WebSearchItem(query='ANU WHS Framework', reason='To gather specific information about the Australian National University (ANU) Workplace Health and Safety (WHS) Framework.'), WebSearchItem(query='ANU health and safety policies', reason='To understand the policies implemented by ANU regarding workplace health and safety.'), WebSearchItem(query='Australian National University WHS policies 2023', reason='To find the most recent updates or changes in the WHS policies of ANU for the current year.')]


In [47]:
INSTRUCTIONS = f"""You are able to send a nicely formatted HTML email based on a detailed report.
You will be provided with a detailed report. You should use your tool to send one email, providing the 
report converted into clean, well presented HTML with an appropriate subject line."""

email_agent = Agent(
    name = "Email_agent",
    instructions = INSTRUCTIONS,
    tools = [send_email],
    model = "gpt-4o-mini"
)



In [48]:
#Search Agent
INSTRUCTIONS = "You are a research assistant. Given a search term, you search the web for that term and \
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 \
words. Capture the main points. Write succintly, no need to have complete sentences or good \
grammar. This will be consumed by someone synthesizing a report, so it's vital you capture the \
essence and ignore any fluff. Do not include any additional commentary other than the summary itself."

search_agent = Agent(
    name="Search agent",
    instructions=INSTRUCTIONS,
    tools=[WebSearchTool(search_context_size="low")],
    model="gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="required"),
)

In [50]:
message = "ANU's WHS Framework"

with trace("Search"):
    result = await Runner.run(search_agent, message)

display(Markdown(result.final_output))

ANU's Work Health and Safety (WHS) Framework is designed to ensure a safe and healthy environment for all staff, students, and visitors. It aligns with the Work Health and Safety Act and Regulations 2011 (Cth) and incorporates the National Self-Insurer WHS Audit Tool (NAT). The framework encompasses all university activities, including teaching, research, and administrative functions. ([services.anu.edu.au](https://services.anu.edu.au/files/guidance/WHS%20Management%20System%20Manual%20July_%202018%20%28Final%29_Updated%20Oct%202018%20%28Final%29..._0.pdf?utm_source=openai))

The WHS Management System Manual outlines the framework's structure, covering elements such as health and safety policy, planning, implementation, measurement and evaluation, and management review. This comprehensive approach ensures that WHS practices are integrated into all university operations. ([services.anu.edu.au](https://services.anu.edu.au/files/guidance/WHS%20Management%20System%20Manual%20July_%202018%20%28Final%29_Updated%20Oct%202018%20%28Final%29..._0.pdf?utm_source=openai))

To support the framework, ANU provides various resources, including the WHS Management System Handbook, which offers practical guidance for implementing WHS practices. Additionally, the university offers training programs accessible through platforms like HORUS and Pulse, covering topics such as WHS induction, risk management, and first aid. ([services.anu.edu.au](https://services.anu.edu.au/human-resources/health-safety?utm_source=openai))

For reporting incidents or hazards, ANU utilizes the Figtree system, allowing staff and students to notify the Safety and Wellbeing Team or ANU Security. ([services.anu.edu.au](https://services.anu.edu.au/human-resources/health-safety?utm_source=openai)) 

In [51]:
INSTRUCTIONS = (
    "You are a senior researcher tasked with writing a cohesive report for a research query. "
    "You will be provided with the original query, and some initial research done by a research assistant.\n"
    "You should first come up with an outline for the report that describes the structure and "
    "flow of the report. Then, generate the report and return that as your final output.\n"
    "The final output should be in markdown format, and it should be lengthy and detailed. Aim "
    "for 5-10 pages of content, at least 1000 words."
)

class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")

    markdown_report: str = Field(description="The final report")

    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(
    name="WriterAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=ReportData,
)

In [ ]:
async def plan_searches(query : str):
    """ Use the planner_agent to plan which searches to run for the query """
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    print(f"Will perform {len(result.final_output.searches)} searches")
    return result.final_output

async def perform_searches(search_plan : WebSearchPlanner):
    """ Call search() for each item in the search plan """
    print("Searching...")
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results
async def search(item: WebSearchItem):
    """Use the search agent to run a web search for each item in the search plan"""